In [1]:

import sqlite3
import json
from pathlib import Path


# --------------------------------------------------
# Configuration
# --------------------------------------------------

RAG_DB = "/content/drive/MyDrive/MAI NLP WS26_27/rag_evaluation.db"
JUDGE_DB = "/content/drive/MyDrive/MAI NLP WS26_27/judge_evaluation.db"
OUTPUT_DB = "/content/drive/MyDrive/MAI NLP WS26_27/merged_evaluation.db"

RAG_TABLE = "rag_results"
JUDGE_TABLE = "evaluation_results"
OUTPUT_TABLE = "qwen3_5_4B_eval_results"


# --------------------------------------------------
# Database merging function
# --------------------------------------------------

def merge_databases():

    # Check that input databases exist
    if not Path(RAG_DB).exists():
        raise FileNotFoundError(f"Database not found: {RAG_DB}")

    if not Path(JUDGE_DB).exists():
        raise FileNotFoundError(f"Database not found: {JUDGE_DB}")

    # Connect to the RAG database
    rag_conn = sqlite3.connect(RAG_DB)
    rag_conn.row_factory = sqlite3.Row

    # Attach the judge database
    rag_conn.execute(
        "ATTACH DATABASE ? AS judge_db",
        (JUDGE_DB,)
    )

    # Create output database
    output_conn = sqlite3.connect(OUTPUT_DB)

    try:

        # --------------------------------------------------
        # Create merged table
        # --------------------------------------------------

        output_conn.execute(f"""
            CREATE TABLE IF NOT EXISTS {OUTPUT_TABLE} (

                id INTEGER PRIMARY KEY AUTOINCREMENT,

                question_id TEXT,

                type TEXT,
                domains TEXT,
                primary_domain TEXT,

                question TEXT NOT NULL,
                ground_truth TEXT,
                response TEXT,

                fetched_context TEXT,

                is_unanswerable INTEGER,

                metadata TEXT,

                rag_created_at TEXT,

                correctness INTEGER,
                coverage INTEGER,
                consistency INTEGER,
                groundedness INTEGER,
                relevance INTEGER,

                reasons TEXT,
                success INTEGER,

                judge_evaluated_at TEXT
            )
        """)

        # --------------------------------------------------
        # Retrieve and join the records
        # --------------------------------------------------

        query = f"""
            SELECT

                rag.id AS question_id,

                rag.type,
                rag.domains,
                rag.primary_domain,
                rag.question,
                rag.ground_truth,
                rag.response,

                -- Use fetched_context from RAG database
                -- If NULL, fall back to selected_chunks
                COALESCE(
                    rag.fetched_context,
                    judge.selected_chunks
                ) AS fetched_context,

                rag.is_unanswerable,

                rag.metadata AS original_metadata,

                rag.created_at AS rag_created_at,

                judge.rag_algorithm,
                judge.top_k,
                judge.reranking_method,

                judge.correctness,
                judge.coverage,
                judge.consistency,
                judge.groundedness,
                judge.relevance,

                judge.reasons,
                judge.success,

                judge.created_at AS judge_evaluated_at

            FROM main.{RAG_TABLE} AS rag

            LEFT JOIN judge_db.{JUDGE_TABLE} AS judge

                ON rag.id = judge.question_id

            ORDER BY rag.id
        """

        rows = rag_conn.execute(query).fetchall()

        # --------------------------------------------------
        # Prepare merged records
        # --------------------------------------------------

        merged_records = []

        for row in rows:

            # Preserve existing metadata
            if row["original_metadata"]:

                try:
                    metadata = json.loads(row["original_metadata"])

                    # Ensure metadata is a dictionary
                    if not isinstance(metadata, dict):
                        metadata = {
                            "original_metadata": metadata
                        }

                except (json.JSONDecodeError, TypeError):
                    metadata = {
                        "original_metadata": row["original_metadata"]
                    }

            else:
                metadata = {}

            # Add RAG configuration to metadata
            if row["rag_algorithm"] is not None:
                metadata["rag_algorithm"] = row["rag_algorithm"]

            if row["top_k"] is not None:
                metadata["top_k"] = row["top_k"]

            if row["reranking_method"] is not None:
                metadata["reranking_method"] = row["reranking_method"]

            metadata_json = json.dumps(
                metadata,
                ensure_ascii=False
            )

            merged_records.append((

                row["question_id"],

                row["type"],
                row["domains"],
                row["primary_domain"],

                row["question"],
                row["ground_truth"],
                row["response"],

                row["fetched_context"],

                row["is_unanswerable"],

                metadata_json,

                row["rag_created_at"],

                row["correctness"],
                row["coverage"],
                row["consistency"],
                row["groundedness"],
                row["relevance"],

                row["reasons"],
                row["success"],

                row["judge_evaluated_at"]
            ))

        # --------------------------------------------------
        # Insert records into merged database
        # --------------------------------------------------

        insert_query = f"""
            INSERT INTO {OUTPUT_TABLE} (

                question_id,

                type,
                domains,
                primary_domain,

                question,
                ground_truth,
                response,

                fetched_context,

                is_unanswerable,

                metadata,

                rag_created_at,

                correctness,
                coverage,
                consistency,
                groundedness,
                relevance,

                reasons,
                success,

                judge_evaluated_at

            )
            VALUES (
                ?, ?, ?, ?, ?, ?, ?, ?, ?, ?,
                ?, ?, ?, ?, ?, ?, ?, ?, ?
            )
        """

        with output_conn:
            output_conn.executemany(
                insert_query,
                merged_records
            )

        # --------------------------------------------------
        # Print summary
        # --------------------------------------------------

        total_merged = output_conn.execute(
            f"SELECT COUNT(*) FROM {OUTPUT_TABLE}"
        ).fetchone()[0]

        matched = sum(
            1 for row in merged_records
            if row[11] is not None
        )

        print("\nDatabase merging completed successfully!")
        print(f"Output database: {OUTPUT_DB}")
        print(f"Total RAG records: {len(rows)}")
        print(f"Total merged records: {total_merged}")
        print(f"Records with judge evaluations: {matched}")
        print(f"Records without judge evaluations: {total_merged - matched}")

    finally:

        output_conn.close()
        rag_conn.close()


# --------------------------------------------------
# Run
# --------------------------------------------------

if __name__ == "__main__":
    merge_databases()



Database merging completed successfully!
Output database: /content/drive/MyDrive/MAI NLP WS26_27/merged_evaluation.db
Total RAG records: 100
Total merged records: 100
Records with judge evaluations: 100
Records without judge evaluations: 0
